# Assignment 5: Instructional Fine-Tuning of LLaMA for NER

**Name:** Ming-Hsiang Lee  
**Dataset:** CoNLL-2003 (local files)  
**Model:** TinyLlama/TinyLlama-1.1B-Chat-v1.0 (LLaMA 2 architecture, freely accessible)  
**Methods:** Full Fine-Tuning (required) + LoRA + QLoRA (optional)  

---

## Overview

Each CoNLL-2003 sentence is formatted as a **chat instruction** asking the model to extract named
entities and return them as a JSON array.  Three fine-tuning strategies are compared:

1. **Full Fine-Tuning** — all 1.1 B parameters updated
2. **LoRA** — only ~4 M low-rank adapter parameters updated; base weights frozen
3. **QLoRA** — same adapters, but base weights quantized to 4-bit NF4 (GPU only)

Entity types: `PER`, `ORG`, `LOC`, `MISC`

> **Recommended runtime:** Google Colab with T4 GPU  
> Runtime → Change runtime type → GPU (T4)

## Section 1: Setup

In [1]:
# Install required packages
# Uncomment on a fresh Colab environment
!pip install transformers peft accelerate seqeval psutil bitsandbytes -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00


### Colab Setup — Upload CoNLL-2003 Files

Before running the notebook on Colab, upload the three CoNLL-2003 data files.

**Recommended: upload the whole folder**
1. In the left sidebar click the **Files** icon.
2. Click **Upload to session storage**.
3. Select (or drag) the entire `conll2003/` folder — it contains `eng.train`, `eng.testa`, `eng.testb`.
4. After upload the folder appears at `/content/conll2003/`.
5. Run the notebook — path resolution is automatic.

**Alternative: upload files individually**
1. Upload `eng.train`, `eng.testa`, `eng.testb` directly (they land in `/content/`).
2. In the Config cell, add `"/content"` as the first entry in `CONLL_CANDIDATES`  
   (or simply set `CONLL_DIR = Path("/content")` after the candidate loop).

**Alternative: mount Google Drive**
```python
from google.colab import drive
drive.mount('/content/drive')
# Then add your Drive path to CONLL_CANDIDATES, e.g.:
# "/content/drive/MyDrive/conll2003"
```

In [2]:
# ---- COLAB ONLY: upload CoNLL-2003 files via the Files panel ----
# Run this cell to trigger the upload dialog (alternative to the sidebar).
# Skip this cell when running locally.

import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import os
    # Check if files were already uploaded
    colab_conll = "/content/conll2003"
    if not os.path.exists(colab_conll):
        print("CoNLL folder not found at /content/conll2003.")
        print("Please upload the conll2003/ folder via the Files panel (left sidebar).")
        print("Expected files: eng.train, eng.testa, eng.testb")
    else:
        files = os.listdir(colab_conll)
        print(f"Found at {colab_conll}: {files}")
else:
    print("Running locally — skipping Colab upload check.")


Found at /content/conll2003: ['eng.testb', 'eng.train', 'eng.testa']


## Section 2: Imports and Configuration

In [3]:
import os
# Reduce GPU memory fragmentation (helps near-limit workloads)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import gc
import json
import time
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import Counter

import psutil
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    set_seed,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    prepare_model_for_kbit_training,
)
from transformers import BitsAndBytesConfig
from seqeval.metrics import (
    classification_report as seqeval_report,
    f1_score        as seqeval_f1,
    precision_score as seqeval_precision,
    recall_score    as seqeval_recall,
)

warnings.filterwarnings("ignore")
set_seed(42)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory: {:.1f} GB".format(
        torch.cuda.get_device_properties(0).total_memory / 1e9
    ))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


In [4]:
# ============================================================
# CONFIGURATION
# ============================================================

# TinyLlama uses the LLaMA 2 architecture and is freely accessible
# on HuggingFace without a special gating agreement.
# Swap with "meta-llama/Llama-3.2-1B-Instruct" if you have HF access.
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# FAST_MODE=True: quick smoke-test (500 train samples, 100 test)
# FAST_MODE=False: full training run (recommended for final submission)
FAST_MODE = True

# Dataset limits per split
FAST_TRAIN_LIMIT = 500
FAST_VAL_LIMIT   = 100
FAST_TEST_LIMIT  = 100
FULL_TRAIN_LIMIT = None   # all ~14 k training sentences
FULL_VAL_LIMIT   = None
FULL_TEST_LIMIT  = 500    # cap inference for speed

# Shared training hyperparameters
TRAIN_EPOCHS  = 3
LEARNING_RATE = 2e-5   # Full FT learning rate
LORA_LR       = 1e-4   # LoRA typically benefits from a higher LR
BATCH_SIZE    = 4
GRAD_ACCUM    = 4      # effective batch size = 16
MAX_SEQ_LEN   = 256    # max tokens per input sample
MAX_NEW_TOKENS = 80    # max tokens generated at inference
WARMUP_STEPS  = 50
WEIGHT_DECAY  = 0.01

# LoRA hyperparameters
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# QLoRA hyperparameters (4-bit NF4 quantization + LoRA adapters)
# Same LoRA adapter config is reused; only the base model changes.
QLORA_QUANT_TYPE       = "nf4"   # NormalFloat4 — best quality at 4-bit
QLORA_DOUBLE_QUANT     = True    # quantize the quantization constants too
QLORA_COMPUTE_DTYPE    = torch.float16

# Output dirs
OUTPUT_DIR_FULL  = "./ckpt_full"
OUTPUT_DIR_LORA  = "./ckpt_lora"
OUTPUT_DIR_QLORA = "./ckpt_qlora"

# Apply limits based on mode
TRAIN_LIMIT = FAST_TRAIN_LIMIT if FAST_MODE else FULL_TRAIN_LIMIT
VAL_LIMIT   = FAST_VAL_LIMIT   if FAST_MODE else FULL_VAL_LIMIT
TEST_LIMIT  = FAST_TEST_LIMIT  if FAST_MODE else FULL_TEST_LIMIT

# CoNLL-2003 path (try both from repo root and from notebook dir)
# ---- CoNLL-2003 path resolution ----------------------------------------
# LOCAL (default): files at  ../Assignment2_Name_Entity_Recognition/conll2003/
# COLAB option A : upload the conll2003 FOLDER to Colab → /content/conll2003/
# COLAB option B : upload eng.train / eng.testa / eng.testb individually
#                  → set CONLL_DIR manually to Path("/content")
CONLL_CANDIDATES = [
    "/content/conll2003",                              # Colab: folder upload
    "Assignment2_Name_Entity_Recognition/conll2003",   # Local: from repo root
    "../Assignment2_Name_Entity_Recognition/conll2003",# Local: from notebook dir
]
CONLL_DIR = None
for candidate in CONLL_CANDIDATES:
    if Path(candidate).exists():
        CONLL_DIR = Path(candidate)
        break
if CONLL_DIR is None:
    raise FileNotFoundError(
        f"CoNLL-2003 directory not found. Tried: {CONLL_CANDIDATES}"
    )

# Device / dtype
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.float16 if torch.cuda.is_available() else torch.float32
USE_FP16 = torch.cuda.is_available()

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Model      : {MODEL_NAME}")
print(f"CoNLL dir  : {CONLL_DIR}")
print(f"Device     : {DEVICE}  dtype={DTYPE}")
print(f"Limits     : train={TRAIN_LIMIT}, val={VAL_LIMIT}, test={TEST_LIMIT}")

FAST_MODE  : True
Model      : TinyLlama/TinyLlama-1.1B-Chat-v1.0
CoNLL dir  : /content/conll2003
Device     : cuda  dtype=torch.float16
Limits     : train=500, val=100, test=100


## Section 3: Data Loading (CoNLL-2003)

CoNLL-2003 column format per token: `WORD  POS  CHUNK  NER`  
Standard splits used: `eng.train` (train), `eng.testa` (validation), `eng.testb` (test).

In [5]:
def load_conll(path: Path) -> List[Tuple[List[str], List[str]]]:
    """
    Parse a CoNLL-2003 file.
    Returns a list of (tokens, bio_labels) tuples.
    The NER label is the last (4th) whitespace-separated column.
    """
    sentences = []
    tokens, labels = [], []
    with open(path, encoding="utf-8") as fh:
        for raw_line in fh:
            line = raw_line.rstrip()
            if not line or line.startswith("-DOCSTART-"):
                if tokens:
                    sentences.append((tokens[:], labels[:]))
                    tokens.clear()
                    labels.clear()
            else:
                parts = line.split()
                tokens.append(parts[0])
                labels.append(parts[-1])  # NER is last column
    if tokens:
        sentences.append((tokens[:], labels[:]))
    return sentences


# Load splits
train_data = load_conll(CONLL_DIR / "eng.train")
val_data   = load_conll(CONLL_DIR / "eng.testa")
test_data  = load_conll(CONLL_DIR / "eng.testb")

print("Full split sizes:")
print(f"  Train: {len(train_data):,} sentences")
print(f"  Val:   {len(val_data):,} sentences")
print(f"  Test:  {len(test_data):,} sentences")

# Apply limits
if TRAIN_LIMIT: train_data = train_data[:TRAIN_LIMIT]
if VAL_LIMIT:   val_data   = val_data[:VAL_LIMIT]
if TEST_LIMIT:  test_data  = test_data[:TEST_LIMIT]

print(f"\nUsing: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}")

# Label distribution
all_labels   = [lbl for _, lbls in train_data for lbl in lbls]
label_counts = Counter(all_labels)
print("\nLabel distribution (train subset):")
for lbl, cnt in sorted(label_counts.items()):
    print(f"  {lbl:12s}: {cnt}")

# Preview
print("\nSample (index 2):")
print("  Tokens:", train_data[2][0])
print("  Labels:", train_data[2][1])

Full split sizes:
  Train: 14,041 sentences
  Val:   3,250 sentences
  Test:  3,453 sentences

Using: train=500, val=100, test=100

Label distribution (train subset):
  B-LOC       : 338
  B-MISC      : 164
  B-ORG       : 156
  B-PER       : 287
  I-LOC       : 33
  I-MISC      : 46
  I-ORG       : 107
  I-PER       : 221
  O           : 5832

Sample (index 2):
  Tokens: ['BRUSSELS', '1996-08-22']
  Labels: ['B-LOC', 'O']


## Section 4: Instructional Format

Each sentence is wrapped in TinyLlama's chat template:

```
<|system|>
{system_prompt}</s>
<|user|>
Text: {sentence}</s>
<|assistant|>
[{"entity": "EU", "type": "ORG"}, ...]</s>   ← training target only
```

During **training**: full prompt including the JSON output is tokenized;  
instruction tokens receive `label = -100` (excluded from loss).  
During **inference**: only the instruction prefix is fed; the model generates the JSON answer.

In [6]:
SYSTEM_PROMPT = (
    "You are a named entity recognition (NER) expert. "
    "Extract all named entities from the given text and return them as a JSON array. "
    "Each element must have exactly two keys: "
    "'entity' (the exact text span) and "
    "'type' (one of: PER, ORG, LOC, MISC). "
    "Return [] if no named entities are present. "
    "Output ONLY the JSON array. No explanation."
)


def bio_to_entities(tokens: List[str], labels: List[str]) -> List[Dict]:
    """Convert a BIO-tagged sentence to a list of {entity, type} dicts."""
    entities   = []
    cur_toks   = []
    cur_type   = None

    for token, label in zip(tokens, labels):
        if label.startswith("B-"):
            if cur_toks:
                entities.append({"entity": " ".join(cur_toks), "type": cur_type})
            cur_toks = [token]
            cur_type = label[2:]
        elif label.startswith("I-") and cur_toks:
            cur_toks.append(token)
        else:
            if cur_toks:
                entities.append({"entity": " ".join(cur_toks), "type": cur_type})
            cur_toks, cur_type = [], None

    if cur_toks:
        entities.append({"entity": " ".join(cur_toks), "type": cur_type})
    return entities


def format_prompt(
    tokens: List[str],
    labels: Optional[List[str]] = None,
    add_output: bool = False,
) -> str:
    """
    Build a TinyLlama chat-format prompt string.
    If add_output=True, the JSON answer is appended (used for training).
    """
    text   = " ".join(tokens)
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
        f"<|user|>\nText: {text}</s>\n"
        f"<|assistant|>\n"
    )
    if add_output and labels is not None:
        entities = bio_to_entities(tokens, labels)
        prompt  += json.dumps(entities, ensure_ascii=False) + "</s>"
    return prompt


# Preview
print("=== Training format (first 2 samples) ===")
for tokens, labels in train_data[:2]:
    ents   = bio_to_entities(tokens, labels)
    prompt = format_prompt(tokens, labels=labels, add_output=True)
    print(f"Tokens  : {tokens}")
    print(f"Entities: {ents}")
    print(f"Prompt  :\n{prompt}")
    print("-" * 60)

=== Training format (first 2 samples) ===
Tokens  : ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Entities: [{'entity': 'EU', 'type': 'ORG'}, {'entity': 'German', 'type': 'MISC'}, {'entity': 'British', 'type': 'MISC'}]
Prompt  :
<|system|>
You are a named entity recognition (NER) expert. Extract all named entities from the given text and return them as a JSON array. Each element must have exactly two keys: 'entity' (the exact text span) and 'type' (one of: PER, ORG, LOC, MISC). Return [] if no named entities are present. Output ONLY the JSON array. No explanation.</s>
<|user|>
Text: EU rejects German call to boycott British lamb .</s>
<|assistant|>
[{"entity": "EU", "type": "ORG"}, {"entity": "German", "type": "MISC"}, {"entity": "British", "type": "MISC"}]</s>
------------------------------------------------------------
Tokens  : ['Peter', 'Blackburn']
Entities: [{'entity': 'Peter Blackburn', 'type': 'PER'}]
Prompt  :
<|system|>
You are a named entity re

## Section 5: Tokenizer and Dataset

In [7]:
# Shared tokenizer for both experiments
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocabulary size : {tokenizer.vocab_size:,}")
print(f"Pad token       : '{tokenizer.pad_token}'  (id={tokenizer.pad_token_id})")
print(f"EOS token       : '{tokenizer.eos_token}'  (id={tokenizer.eos_token_id})")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Vocabulary size : 32,000
Pad token       : '</s>'  (id=2)
EOS token       : '</s>'  (id=2)


In [8]:
class NERInstructionDataset(Dataset):
    """
    Tokenizes (tokens, bio_labels) pairs for causal LM fine-tuning.
    Instruction tokens are masked to -100; loss is computed only on
    the JSON output tokens.
    """

    def __init__(
        self,
        data: List[Tuple[List[str], List[str]]],
        tokenizer,
        max_length: int = MAX_SEQ_LEN,
    ):
        self.examples = []
        skipped = 0

        for tokens, labels in data:
            full_prompt = format_prompt(tokens, labels=labels, add_output=True)
            inst_prompt = format_prompt(tokens, add_output=False)

            full_enc = tokenizer(
                full_prompt, truncation=True,
                max_length=max_length, return_tensors="pt",
            )
            inst_enc = tokenizer(
                inst_prompt, truncation=True,
                max_length=max_length, return_tensors="pt",
            )

            input_ids = full_enc["input_ids"][0]
            attn_mask = full_enc["attention_mask"][0]
            inst_len  = inst_enc["input_ids"].shape[1]

            # Mask instruction tokens from loss
            lbl = input_ids.clone()
            lbl[:inst_len] = -100

            # Skip if output was fully truncated
            if (lbl != -100).sum().item() == 0:
                skipped += 1
                continue

            self.examples.append(
                {"input_ids": input_ids, "attention_mask": attn_mask, "labels": lbl}
            )

        if skipped:
            print(f"  [!] Skipped {skipped} samples (output fully truncated)")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def collate_fn(batch):
    """Right-pad all sequences in the batch to the same length."""
    max_len = max(item["input_ids"].shape[0] for item in batch)
    pad_id  = tokenizer.pad_token_id
    return {
        "input_ids": torch.stack([
            F.pad(item["input_ids"],
                  (0, max_len - len(item["input_ids"])), value=pad_id)
            for item in batch
        ]),
        "attention_mask": torch.stack([
            F.pad(item["attention_mask"],
                  (0, max_len - len(item["attention_mask"])), value=0)
            for item in batch
        ]),
        "labels": torch.stack([
            F.pad(item["labels"],
                  (0, max_len - len(item["labels"])), value=-100)
            for item in batch
        ]),
    }


print("Building datasets ...")
train_dataset = NERInstructionDataset(train_data, tokenizer)
print(f"  Train : {len(train_dataset)} samples")
val_dataset = NERInstructionDataset(val_data, tokenizer)
print(f"  Val   : {len(val_dataset)} samples")

# Test set stays as raw (tokens, labels) pairs — used for generation evaluation
test_sentences = test_data
print(f"  Test  : {len(test_sentences)} sentence pairs (raw, for generation)")

# Sanity check
s = train_dataset[0]
print(f"\nSample[0] shapes  : input_ids={s['input_ids'].shape}, labels={s['labels'].shape}")
print(f"Non-masked labels : {(s['labels'] != -100).sum().item()} tokens")

Building datasets ...
  Train : 500 samples
  Val   : 100 samples
  Test  : 100 sentence pairs (raw, for generation)

Sample[0] shapes  : input_ids=torch.Size([174]), labels=torch.Size([174])
Non-masked labels : 48 tokens


## Section 6: Resource Tracking and Evaluation Utilities

In [9]:
# -------------------------------------------------- Resource tracker
class ResourceTracker:
    """Context manager: measures wall-clock time and peak GPU/CPU memory."""

    def __init__(self, name: str):
        self.name  = name
        self.stats = {}

    def __enter__(self):
        self.t0 = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        return self

    def __exit__(self, *args):
        elapsed  = time.time() - self.t0
        cpu_mb   = psutil.Process().memory_info().rss / 1e6
        gpu_peak = (
            torch.cuda.max_memory_allocated() / 1e6
            if torch.cuda.is_available() else 0
        )
        self.stats = {
            "name":        self.name,
            "time_s":      round(elapsed, 1),
            "cpu_ram_mb":  round(cpu_mb, 0),
            "gpu_peak_mb": round(gpu_peak, 0),
        }
        print(
            f"[{self.name}]  "
            f"Time: {elapsed:.1f}s  |  "
            f"CPU RAM: {cpu_mb:.0f} MB  |  "
            f"GPU peak: {gpu_peak:.0f} MB"
        )


# -------------------------------------------------- Parameter counter
def count_params(model) -> Tuple[int, int]:
    """Return (total, trainable) parameter counts."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# -------------------------------------------------- Output parser
def parse_entities(text: str) -> List[Dict]:
    """Extract an entity list from raw model-generated text."""
    text = text.strip()
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parsed
    except json.JSONDecodeError:
        pass
    # Fallback: find first JSON array via regex
    m = re.search(r"\[.*?\]", text, re.DOTALL)
    if m:
        try:
            parsed = json.loads(m.group())
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass
    return []


# -------------------------------------------------- BIO aligner
def entities_to_bio(
    tokens: List[str], entities: List[Dict]
) -> List[str]:
    """
    Re-align predicted entity dicts to the original token sequence
    using case-insensitive exact token-span matching.
    """
    VALID = {"PER", "ORG", "LOC", "MISC"}
    tags  = ["O"] * len(tokens)
    tl    = [t.lower() for t in tokens]

    for ent in entities:
        if not isinstance(ent, dict):  # skip malformed items (strings, etc.)
            continue
        et   = ent.get("entity", "")
        etype = ent.get("type",   "")
        if not et or etype not in VALID:
            continue
        etoks = [t.lower() for t in et.split()]
        n     = len(etoks)
        for i in range(len(tokens) - n + 1):
            if tl[i : i + n] == etoks:
                tags[i] = f"B-{etype}"
                for j in range(1, n):
                    tags[i + j] = f"I-{etype}"
                break
    return tags


# -------------------------------------------------- Evaluation function
def evaluate_model(
    model,
    tokenizer,
    test_sentences: List[Tuple[List[str], List[str]]],
    desc: str = "Model",
) -> Dict:
    """
    Generate predictions and compute entity-level seqeval metrics.
    """
    model.eval()
    device = next(model.parameters()).device

    gold_bio, pred_bio = [], []
    parse_ok = 0
    t0 = time.time()

    for tokens, labels in test_sentences:
        prompt = format_prompt(tokens, add_output=False)
        enc = tokenizer(
            prompt, return_tensors="pt", truncation=True,
            max_length=MAX_SEQ_LEN - MAX_NEW_TOKENS,
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        gen_ids     = gen[0][enc["input_ids"].shape[1] :]
        output_text = tokenizer.decode(gen_ids, skip_special_tokens=True)

        pred_entities = parse_entities(output_text)
        if isinstance(pred_entities, list):
            parse_ok += 1

        pred_tags = entities_to_bio(tokens, pred_entities)
        gold_bio.append(labels)
        pred_bio.append(pred_tags)

    elapsed   = time.time() - t0
    precision = seqeval_precision(gold_bio, pred_bio)
    recall    = seqeval_recall(gold_bio, pred_bio)
    f1        = seqeval_f1(gold_bio, pred_bio)
    report    = seqeval_report(gold_bio, pred_bio)

    print(f"\n{'='*55}")
    print(f" Evaluation: {desc}")
    print(f"{'='*55}")
    print(f"Sentences evaluated : {len(test_sentences)}")
    print(f"Parse success rate  : {parse_ok}/{len(test_sentences)} "
          f"({100*parse_ok/len(test_sentences):.1f}%)")
    print(f"Inference time      : {elapsed:.1f}s")
    print(f"Entity Precision    : {precision:.4f}")
    print(f"Entity Recall       : {recall:.4f}")
    print(f"Entity F1           : {f1:.4f}")
    print(f"\nPer-class report:\n{report}")

    return {
        "precision":        round(precision, 4),
        "recall":           round(recall,    4),
        "f1":               round(f1,        4),
        "parse_rate":       round(parse_ok / len(test_sentences), 4),
        "inference_time_s": round(elapsed,   1),
    }


# Global results dictionary
results = {}
print("Utilities ready.")

Utilities ready.


## Section 7: Experiment 1 — Full Fine-Tuning

All 1.1 B parameters are updated.  
Gradient checkpointing is enabled to fit within T4 VRAM (~16 GB).

| Hyperparameter | Value |
|---|---|
| Optimizer | AdamW |
| Learning rate | 2e-5 |
| Effective batch size | 16 (4 × 4 accum) |
| Epochs | 3 |
| Warmup steps | 50 |
| Weight decay | 0.01 |
| Precision | fp16 |

In [10]:
print(f"Loading model for Full Fine-Tuning: {MODEL_NAME}")

model_full = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,  # fp32 required for AMP (fp16 training arg)
    device_map="auto" if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    model_full = model_full.to(DEVICE)

model_full.gradient_checkpointing_enable()
model_full.enable_input_require_grads()

total_full, trainable_full = count_params(model_full)
print(f"Total parameters     : {total_full:,}")
print(f"Trainable parameters : {trainable_full:,}  ({100*trainable_full/total_full:.1f}%)")

# Full FT in fp32 is memory-heavy on T4 (14.56 GB).
# In full mode, halve per-device batch and double accumulation
# so effective batch stays 16 but peak activation memory is halved.
_ft_batch = 2 if (torch.cuda.is_available() and not FAST_MODE) else BATCH_SIZE
_ft_accum = 8 if (torch.cuda.is_available() and not FAST_MODE) else GRAD_ACCUM

training_args_full = TrainingArguments(
    output_dir                  = OUTPUT_DIR_FULL,
    num_train_epochs            = TRAIN_EPOCHS,
    per_device_train_batch_size = _ft_batch,
    gradient_accumulation_steps = _ft_accum,
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    warmup_steps                = WARMUP_STEPS,
    fp16                        = USE_FP16,
    logging_steps               = 20,
    save_strategy               = "no",
    eval_strategy               = "no",
    dataloader_num_workers      = 0,
    report_to                   = "none",
    use_cpu                     = not torch.cuda.is_available(),
    optim                       = "adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
)

trainer_full = Trainer(
    model         = model_full,
    args          = training_args_full,
    train_dataset = train_dataset,
    data_collator = collate_fn,
)

print("\nStarting Full Fine-Tuning ...")
tracker_full = ResourceTracker("Full Fine-Tuning")
with tracker_full:
    train_result_full = trainer_full.train()

print(f"\nFinal training loss: {train_result_full.training_loss:.4f}")

results["full_ft"] = dict(tracker_full.stats)
results["full_ft"]["trainable_params"] = trainable_full
results["full_ft"]["trainable_pct"]    = 100.0
results["full_ft"]["train_loss"]       = round(train_result_full.training_loss, 4)

Loading model for Full Fine-Tuning: TinyLlama/TinyLlama-1.1B-Chat-v1.0


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total parameters     : 1,100,048,384
Trainable parameters : 1,100,048,384  (100.0%)

Starting Full Fine-Tuning ...


Step,Training Loss
20,0.301231
40,0.087036
60,0.049247
80,0.030976


[Full Fine-Tuning]  Time: 258.4s  |  CPU RAM: 3056 MB  |  GPU peak: 14055 MB

Final training loss: 0.1006


In [11]:
# Evaluate Full Fine-Tuning model
print("Evaluating Full Fine-Tuning model on test set ...")
metrics_full = evaluate_model(model_full, tokenizer, test_sentences, "Full Fine-Tuning")
results.setdefault("full_ft", {}).update(metrics_full)

print("\nFull FT result summary:")
print(json.dumps(results["full_ft"], indent=2))

Evaluating Full Fine-Tuning model on test set ...

 Evaluation: Full Fine-Tuning
Sentences evaluated : 100
Parse success rate  : 100/100 (100.0%)
Inference time      : 153.0s
Entity Precision    : 0.9286
Entity Recall       : 0.7189
Entity F1           : 0.8104

Per-class report:
              precision    recall  f1-score   support

         LOC       0.95      0.98      0.96        82
        MISC       0.68      0.77      0.72        22
         ORG       1.00      0.50      0.67         2
         PER       1.00      0.52      0.69       111

   micro avg       0.93      0.72      0.81       217
   macro avg       0.91      0.69      0.76       217
weighted avg       0.95      0.72      0.79       217


Full FT result summary:
{
  "name": "Full Fine-Tuning",
  "time_s": 258.4,
  "cpu_ram_mb": 3056.0,
  "gpu_peak_mb": 14055.0,
  "trainable_params": 1100048384,
  "trainable_pct": 100.0,
  "train_loss": 0.1006,
  "precision": 0.9286,
  "recall": 0.7189,
  "f1": 0.8104,
  "parse_rate":

## Section 8: Experiment 2 — LoRA Fine-Tuning

Low-Rank Adaptation (LoRA) injects trainable rank-decomposition matrices `A` and `B` into the
attention projection layers.  Base model weights are **frozen**.  Only the adapter weights (~4 M)
are updated, drastically reducing optimizer memory.

| Hyperparameter | Value |
|---|---|
| Rank `r` | 16 |
| Alpha | 32 |
| Dropout | 0.05 |
| Target modules | q_proj, v_proj, k_proj, o_proj |
| Learning rate | 1e-4 (5× larger than Full FT) |
| Effective batch size | 16 |
| Epochs | 3 |

In [12]:
# Free GPU memory from Full FT before loading next model
if 'model_full' in globals(): del model_full
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Freed Full FT model from GPU memory.")

print(f"\nLoading base model for LoRA: {MODEL_NAME}")
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto" if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    model_base = model_base.to(DEVICE)

# Attach LoRA adapters
lora_config = LoraConfig(
    r               = LORA_R,
    lora_alpha      = LORA_ALPHA,
    target_modules  = LORA_TARGET_MODULES,
    lora_dropout    = LORA_DROPOUT,
    bias            = "none",
    task_type       = TaskType.CAUSAL_LM,
)
model_lora = get_peft_model(model_base, lora_config)
model_lora.gradient_checkpointing_enable()
model_lora.enable_input_require_grads()

total_lora, trainable_lora = count_params(model_lora)
print(f"Total parameters     : {total_lora:,}")
print(f"Trainable parameters : {trainable_lora:,}  "
      f"({100*trainable_lora/total_lora:.3f}%)")
model_lora.print_trainable_parameters()

training_args_lora = TrainingArguments(
    output_dir                  = OUTPUT_DIR_LORA,
    num_train_epochs            = TRAIN_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LORA_LR,
    weight_decay                = WEIGHT_DECAY,
    warmup_steps                = WARMUP_STEPS,
    fp16                        = USE_FP16,
    logging_steps               = 20,
    save_strategy               = "no",
    eval_strategy               = "no",
    dataloader_num_workers      = 0,
    report_to                   = "none",
    use_cpu                     = not torch.cuda.is_available(),
)

trainer_lora = Trainer(
    model         = model_lora,
    args          = training_args_lora,
    train_dataset = train_dataset,
    data_collator = collate_fn,
)

print("\nStarting LoRA training ...")
tracker_lora = ResourceTracker("LoRA Fine-Tuning")
with tracker_lora:
    train_result_lora = trainer_lora.train()

print(f"\nFinal training loss: {train_result_lora.training_loss:.4f}")

results["lora"] = dict(tracker_lora.stats)
results["lora"]["trainable_params"] = trainable_lora
results["lora"]["trainable_pct"]    = round(100 * trainable_lora / total_lora, 3)
results["lora"]["train_loss"]       = round(train_result_lora.training_loss, 4)

Freed Full FT model from GPU memory.

Loading base model for LoRA: TinyLlama/TinyLlama-1.1B-Chat-v1.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total parameters     : 1,104,553,984
Trainable parameters : 4,505,600  (0.408%)
trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079

Starting LoRA training ...


Step,Training Loss
20,0.581420
40,0.218755
60,0.120454
80,0.085566


[LoRA Fine-Tuning]  Time: 166.5s  |  CPU RAM: 4794 MB  |  GPU peak: 9810 MB

Final training loss: 0.2229


In [13]:
# Evaluate LoRA model
print("Evaluating LoRA model on test set ...")
metrics_lora = evaluate_model(model_lora, tokenizer, test_sentences, "LoRA")
results.setdefault("lora", {}).update(metrics_lora)

print("\nLoRA result summary:")
print(json.dumps(results["lora"], indent=2))

Evaluating LoRA model on test set ...

 Evaluation: LoRA
Sentences evaluated : 100
Parse success rate  : 100/100 (100.0%)
Inference time      : 143.2s
Entity Precision    : 0.8603
Entity Recall       : 0.5392
Entity F1           : 0.6629

Per-class report:
              precision    recall  f1-score   support

         LOC       0.98      0.66      0.79        82
        MISC       0.52      0.50      0.51        22
         ORG       0.00      0.00      0.00         2
         PER       0.93      0.47      0.62       111

   micro avg       0.86      0.54      0.66       217
   macro avg       0.61      0.41      0.48       217
weighted avg       0.90      0.54      0.67       217


LoRA result summary:
{
  "name": "LoRA Fine-Tuning",
  "time_s": 166.5,
  "cpu_ram_mb": 4794.0,
  "gpu_peak_mb": 9810.0,
  "trainable_params": 4505600,
  "trainable_pct": 0.408,
  "train_loss": 0.2229,
  "precision": 0.8603,
  "recall": 0.5392,
  "f1": 0.6629,
  "parse_rate": 1.0,
  "inference_time_s": 143

## Section 9: Experiment 3 — QLoRA Fine-Tuning

QLoRA (Dettmers et al., 2023) combines **4-bit NF4 quantization** of the frozen base model
with **LoRA adapters** trained in fp16.  The base model weights occupy only ~0.55 GB
(vs ~2.2 GB for fp16 LoRA), dramatically reducing GPU memory requirements.

Workflow:
1. Load base model in 4-bit (`BitsAndBytesConfig`)
2. Call `prepare_model_for_kbit_training()` to stabilize layer norms in fp32
3. Attach LoRA adapters with `get_peft_model()` — same config as LoRA experiment
4. Train only adapter weights (base model stays quantized and frozen)

| Hyperparameter | Value |
|---|---|
| Quantization | 4-bit NF4 + double quantization |
| LoRA rank `r` | 16 |
| LoRA alpha | 32 |
| LoRA dropout | 0.05 |
| Target modules | q_proj, v_proj, k_proj, o_proj |
| Learning rate | 1e-4 |
| Effective batch size | 16 |
| Epochs | 3 |


In [14]:
# QLoRA requires CUDA (bitsandbytes does not support CPU).
if not torch.cuda.is_available():
    print("[SKIP] QLoRA requires a CUDA GPU. Skipping this section on CPU.")
    print("       Run on Colab T4 GPU to execute QLoRA.")
    results['qlora'] = {'f1': 'N/A', 'precision': 'N/A', 'recall': 'N/A',
                        'gpu_peak_mb': 0, 'time_s': 0, 'train_loss': 'N/A',
                        'trainable_params': 4194304, 'trainable_pct': 0.381,
                        'parse_rate': 'N/A', 'inference_time_s': 'N/A'}
else:
    # Free LoRA model from GPU memory before loading QLoRA
    if 'model_lora' in globals(): del model_lora
    if 'model_base' in globals(): del model_base
    gc.collect()
    torch.cuda.empty_cache()
    print("Freed LoRA model from GPU memory.")
 # ── QLoRA: 4-bit quantization config ────────────────────────────────────
    bnb_config = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_quant_type        = QLORA_QUANT_TYPE,       # "nf4"
        bnb_4bit_compute_dtype     = QLORA_COMPUTE_DTYPE,    # float16
        bnb_4bit_use_double_quant  = QLORA_DOUBLE_QUANT,     # True
    )

    print(f"Loading 4-bit quantized model: {MODEL_NAME}")
    model_qlora_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config = bnb_config,
        device_map          = "auto" if torch.cuda.is_available() else None,
    )

    # Prepare quantized model for training (casts layer norms to fp32,
    # enables gradient checkpointing safely)
    model_qlora_base = prepare_model_for_kbit_training(
        model_qlora_base, use_gradient_checkpointing=True
    )

    # Attach the same LoRA adapter config
    lora_config_q = LoraConfig(
        r               = LORA_R,
        lora_alpha      = LORA_ALPHA,
        target_modules  = LORA_TARGET_MODULES,
        lora_dropout    = LORA_DROPOUT,
        bias            = "none",
        task_type       = TaskType.CAUSAL_LM,
    )
    model_qlora = get_peft_model(model_qlora_base, lora_config_q)
    model_qlora.enable_input_require_grads()

    total_q, trainable_q = count_params(model_qlora)
    print(f"Total parameters     : {total_q:,}")
    print(f"Trainable parameters : {trainable_q:,}  "
          f"({100*trainable_q/total_q:.3f}%)")
    model_qlora.print_trainable_parameters()

    training_args_qlora = TrainingArguments(
        output_dir                  = OUTPUT_DIR_QLORA,
        num_train_epochs            = TRAIN_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate               = LORA_LR,
        weight_decay                = WEIGHT_DECAY,
        warmup_steps                = WARMUP_STEPS,
        fp16                        = USE_FP16,
        logging_steps               = 20,
        save_strategy               = "no",
        eval_strategy               = "no",
        dataloader_num_workers      = 0,
        report_to                   = "none",
        optim                       = "paged_adamw_8bit",  # bitsandbytes paged optimizer
    )

    trainer_qlora = Trainer(
        model         = model_qlora,
        args          = training_args_qlora,
        train_dataset = train_dataset,
        data_collator = collate_fn,
    )

    print("\nStarting QLoRA training ...")
    tracker_qlora = ResourceTracker("QLoRA Fine-Tuning")
    with tracker_qlora:
        train_result_qlora = trainer_qlora.train()

    print(f"\nFinal training loss: {train_result_qlora.training_loss:.4f}")

    results["qlora"] = dict(tracker_qlora.stats)
    results["qlora"]["trainable_params"] = trainable_q
    results["qlora"]["trainable_pct"]    = round(100 * trainable_q / total_q, 3)
    results["qlora"]["train_loss"]       = round(train_result_qlora.training_loss, 4)


Freed LoRA model from GPU memory.
Loading 4-bit quantized model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total parameters     : 620,111,872
Trainable parameters : 4,505,600  (0.727%)
trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079

Starting QLoRA training ...


Step,Training Loss
20,0.547530
40,0.215644
60,0.119715
80,0.082965


[QLoRA Fine-Tuning]  Time: 241.1s  |  CPU RAM: 4796 MB  |  GPU peak: 11086 MB

Final training loss: 0.2145


In [15]:
# Evaluate QLoRA model (GPU only)
if not torch.cuda.is_available():
    print("[SKIP] QLoRA evaluation skipped on CPU.")
else:
    print("Evaluating QLoRA model on test set ...")
    metrics_qlora = evaluate_model(model_qlora, tokenizer, test_sentences, "QLoRA")
    results.setdefault("qlora", {}).update(metrics_qlora)
    print("\nQLoRA result summary:")
    print(json.dumps(results["qlora"], indent=2))


Evaluating QLoRA model on test set ...

 Evaluation: QLoRA
Sentences evaluated : 100
Parse success rate  : 100/100 (100.0%)
Inference time      : 224.9s
Entity Precision    : 0.8456
Entity Recall       : 0.5300
Entity F1           : 0.6516

Per-class report:
              precision    recall  f1-score   support

         LOC       0.96      0.65      0.77        82
        MISC       0.46      0.59      0.52        22
         ORG       0.00      0.00      0.00         2
         PER       0.98      0.44      0.61       111

   micro avg       0.85      0.53      0.65       217
   macro avg       0.60      0.42      0.48       217
weighted avg       0.91      0.53      0.66       217


QLoRA result summary:
{
  "name": "QLoRA Fine-Tuning",
  "time_s": 241.1,
  "cpu_ram_mb": 4796.0,
  "gpu_peak_mb": 11086.0,
  "trainable_params": 4505600,
  "trainable_pct": 0.727,
  "train_loss": 0.2145,
  "precision": 0.8456,
  "recall": 0.53,
  "f1": 0.6516,
  "parse_rate": 1.0,
  "inference_time_s": 

## Section 10: Qualitative Inspection

In [16]:
# Show sample predictions — picks whichever model is still in memory
active_model = None
active_name  = None
for _name, _var in [("QLoRA", "model_qlora"), ("LoRA", "model_lora"), ("Full FT", "model_full")]:
    if _var in globals():
        active_model = globals()[_var]
        active_name  = _name
        break

if active_model is None:
    print("No model in memory — re-run the training cell first, then come back here.")
else:
    active_model.eval()
    device = next(active_model.parameters()).device
    print(f"Sample predictions ({active_name} model, first 5 test sentences):")
    print("=" * 70)
    for idx, (tokens, labels) in enumerate(test_sentences[:5]):
        gold_entities = bio_to_entities(tokens, labels)
        prompt = format_prompt(tokens, add_output=False)
        enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                        max_length=MAX_SEQ_LEN - MAX_NEW_TOKENS)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            gen = active_model.generate(
                **enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen_ids = gen[0][enc["input_ids"].shape[1]:]
        output_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
        pred_entities = parse_entities(output_text)
        print(f"[{idx}] Text : {' '.join(tokens)}")
        print(f"      Gold : {gold_entities}")
        print(f"      Pred : {pred_entities}")
        print(f"      Raw  : {output_text[:120]}")
        print()


Sample predictions (QLoRA model, first 5 test sentences):
[0] Text : SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .
      Gold : [{'entity': 'JAPAN', 'type': 'LOC'}, {'entity': 'CHINA', 'type': 'PER'}]
      Pred : [{'entity': 'JAPAN', 'type': 'LOC'}, {'entity': 'CHINA', 'type': 'LOC'}]
      Raw  : [{"entity": "JAPAN", "type": "LOC"}, {"entity": "CHINA", "type": "LOC"}]

[1] Text : Nadim Ladki
      Gold : [{'entity': 'Nadim Ladki', 'type': 'PER'}]
      Pred : []
      Raw  : []

[2] Text : AL-AIN , United Arab Emirates 1996-12-06
      Gold : [{'entity': 'AL-AIN', 'type': 'LOC'}, {'entity': 'United Arab Emirates', 'type': 'LOC'}]
      Pred : [{'entity': 'AL-AIN', 'type': 'LOC'}, {'entity': 'United Arab Emirates', 'type': 'LOC'}]
      Raw  : [{"entity": "AL-AIN", "type": "LOC"}, {"entity": "United Arab Emirates", "type": "LOC"}]

[3] Text : Japan began the defence of their Asian Cup title with a lucky 2-1 win against Syria in a Group C championship match on Friday .
    

## Section 11: Comparison Table and Analysis

In [17]:
# ----------------------------------------------------------------
# RESULTS RECOVERY
# If the kernel was restarted and results{} is empty, fill it here
# from the output you already observed, then run the comparison cell.
# ----------------------------------------------------------------
if "full_ft" not in results:
    results["full_ft"] = {
        # ← paste your Full FT output values
        "trainable_params": 1100048384, "trainable_pct": 100.0,
        "train_loss": None, "time_s": None, "gpu_peak_mb": 0,
        "precision": None, "recall": None, "f1": None, "parse_rate": None,
    }
if "lora" not in results:
    results["lora"] = {
        # ← paste your LoRA output values  (e.g. from the eval printout)
        "trainable_params": 4194304, "trainable_pct": 0.381,
        "train_loss": None, "time_s": None, "gpu_peak_mb": 0,
        "precision": 0.8082, "recall": 0.5438, "f1": 0.6501, "parse_rate": 1.0,
    }
if "qlora" not in results:
    results["qlora"] = {
        # ← QLoRA skipped on CPU
        "trainable_params": 4194304, "trainable_pct": 0.381,
        "train_loss": "N/A", "time_s": "N/A", "gpu_peak_mb": 0,
        "precision": "N/A", "recall": "N/A", "f1": "N/A", "parse_rate": "N/A",
    }
print("results keys:", list(results.keys()))


results keys: ['full_ft', 'lora', 'qlora']


In [18]:
# ----------------------------------------------------------------
# Experiment comparison (3 methods)
# ----------------------------------------------------------------
# Helper: safely fetch from results dict, return "N/A" if key missing
def _r(method, key, fmt=None):
    val = results.get(method, {}).get(key, "N/A")
    if fmt and val != "N/A":
        try:
            return fmt.format(val)
        except Exception:
            pass
    return val

df_compare = pd.DataFrame({
    "Method"         : ["Full Fine-Tuning", "LoRA", "QLoRA"],
    "Trainable Params": [_r("full_ft","trainable_params","{:.0f} M" if _r("full_ft","trainable_params") != "N/A" else None),
                         _r("lora",   "trainable_params"),
                         _r("qlora",  "trainable_params")],
    "Trainable %"    : [_r("full_ft","trainable_pct"),  _r("lora","trainable_pct"),  _r("qlora","trainable_pct")],
    "GPU Peak (MB)"  : [_r("full_ft","gpu_peak_mb"),    _r("lora","gpu_peak_mb"),    _r("qlora","gpu_peak_mb")],
    "Train Time (s)" : [_r("full_ft","time_s"),         _r("lora","time_s"),         _r("qlora","time_s")],
    "Train Loss"     : [_r("full_ft","train_loss"),     _r("lora","train_loss"),     _r("qlora","train_loss")],
    "Precision"      : [_r("full_ft","precision"),      _r("lora","precision"),      _r("qlora","precision")],
    "Recall"         : [_r("full_ft","recall"),         _r("lora","recall"),         _r("qlora","recall")],
    "Entity F1"      : [_r("full_ft","f1"),             _r("lora","f1"),             _r("qlora","f1")],
    "JSON Parse Rate": [_r("full_ft","parse_rate"),     _r("lora","parse_rate"),     _r("qlora","parse_rate")],
})

# Format trainable params nicely
def _fmt_params(v):
    try:
        return f"{float(v)/1e6:.1f} M"
    except Exception:
        return str(v)
df_compare["Trainable Params"] = df_compare["Trainable Params"].apply(_fmt_params)

print("=" * 110)
print("EXPERIMENT COMPARISON")
print("=" * 110)
print(df_compare.to_string(index=False))

# ----------------------------------------------------------------
# Cross-assignment comparison (historical baselines)
# ----------------------------------------------------------------
df_cross = pd.DataFrame({
    "Assignment" : ["A2",   "A2",      "A3",               "A5",               "A5",           "A5"],
    "Method"     : ["CRF",  "BiLSTM",  "BERT (token cls)",
                    "TinyLlama Full FT",  "TinyLlama LoRA",  "TinyLlama QLoRA"],
    "Entity F1"  : [
        0.7905, 0.6347, 0.9108,
        _r("full_ft","f1"), _r("lora","f1"), _r("qlora","f1"),
    ],
})
print("\nCROSS-ASSIGNMENT COMPARISON")
print(df_cross.to_string(index=False))

# Save results
out_path = Path("assignment5_results.json")
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to: {out_path}")


EXPERIMENT COMPARISON
          Method Trainable Params  Trainable %  GPU Peak (MB)  Train Time (s)  Train Loss  Precision  Recall  Entity F1  JSON Parse Rate
Full Fine-Tuning     1100048384 M      100.000        14055.0           258.4      0.1006     0.9286  0.7189     0.8104              1.0
            LoRA            4.5 M        0.408         9810.0           166.5      0.2229     0.8603  0.5392     0.6629              1.0
           QLoRA            4.5 M        0.727        11086.0           241.1      0.2145     0.8456  0.5300     0.6516              1.0

CROSS-ASSIGNMENT COMPARISON
Assignment            Method  Entity F1
        A2               CRF     0.7905
        A2            BiLSTM     0.6347
        A3  BERT (token cls)     0.9108
        A5 TinyLlama Full FT     0.8104
        A5    TinyLlama LoRA     0.6629
        A5   TinyLlama QLoRA     0.6516

Results saved to: assignment5_results.json


## Section 12: Summary

Results from Colab T4 GPU fast-mode run (500 train / 100 val / 100 test sentences).

### Resource Usage

| Method | Trainable Params | GPU Peak (MB) | Train Time (s) | Train Loss |
|---|---:|---:|---:|---:|
| Full Fine-Tuning | 1,100 M (100%) | **14,055** | 258.4 | **0.1006** |
| LoRA | 4.5 M (0.41%) | 9,810 | **166.5** | 0.2229 |
| QLoRA | 4.5 M (0.41%) | 11,086 | 241.1 | 0.2145 |

### NER Performance (entity-level, seqeval)

| Method | Precision | Recall | F1 | Parse Rate |
|---|---:|---:|---:|---:|
| Full Fine-Tuning | **0.9286** | **0.7189** | **0.8104** | 100% |
| LoRA | 0.8603 | 0.5392 | 0.6629 | 100% |
| QLoRA | 0.8456 | 0.5300 | 0.6516 | 100% |

### Per-class F1

| Class | Full FT | LoRA | QLoRA |
|---|---:|---:|---:|
| LOC | **0.96** | 0.79 | 0.77 |
| MISC | **0.72** | 0.51 | 0.52 |
| ORG | **0.67** | 0.00 | 0.00 |
| PER | **0.69** | 0.62 | 0.61 |

ORG (only 2 test entities) and PER suffer from low recall under adapter methods — the model
often recognises the entity text but fails to output it in the JSON array.

### Key Findings

**Accuracy:**
- Full Fine-Tuning achieves the highest F1 (0.8104), substantially above LoRA (0.6629)
  and QLoRA (0.6516). Updating all 1.1 B parameters allows the model to memorise the
  instructional output format quickly — reflected in its very low training loss (0.1006).
- LoRA and QLoRA are close to each other (ΔF1 = 0.011), showing that 4-bit quantisation
  introduces only minor accuracy degradation for adapter-based fine-tuning.
- All three methods achieve 100% JSON parse rate on Colab GPU, meaning the model reliably
  follows the output format once trained.
- Full FT surpasses CRF (0.79) and BiLSTM (0.63) from Assignment 2, and approaches
  BERT token classification (0.9108) from Assignment 3 — impressive for a generative approach.

**Resource efficiency:**
- LoRA uses **30% less GPU memory** than Full FT (9,810 vs 14,055 MB) and is
  **36% faster** (166.5 vs 258.4 s), at a cost of −0.148 F1.
- QLoRA uses **21% less GPU memory** than Full FT but is slower than LoRA (241.1 s)
  due to dequantisation overhead during the forward pass. Its GPU peak (11,086 MB)
  is higher than LoRA's because fp16 dequantisation buffers are allocated during inference.
- LoRA is the most time-efficient adapter method; QLoRA's memory advantage over LoRA is
  smaller than expected on T4 because the 4-bit base model still dequantises to fp16
  for every matrix multiply.

**Trade-off summary:**

| Dimension | Full FT | LoRA | QLoRA |
|---|---|---|---|
| Entity F1 | **0.8104** | 0.6629 | 0.6516 |
| GPU Peak | 14,055 MB | **9,810 MB** | 11,086 MB |
| Train Time | 258.4 s | **166.5 s** | 241.1 s |
| Train Loss | **0.1006** | 0.2229 | 0.2145 |
| Trainable Params | 1,100 M (100%) | 4.5 M (0.41%) | 4.5 M (0.41%) |
| Parse Rate | 100% | 100% | 100% |

**Cross-assignment comparison:**

| Assignment | Method | Entity F1 |
|---|---|---:|
| A2 | CRF | 0.7905 |
| A2 | BiLSTM | 0.6347 |
| A3 | BERT (token classification) | **0.9108** |
| A5 | TinyLlama Full Fine-Tuning | 0.8104 |
| A5 | TinyLlama LoRA | 0.6629 |
| A5 | TinyLlama QLoRA | 0.6516 |

**Conclusion:**
Full Fine-Tuning delivers the best NER accuracy among the three methods and is competitive
with prior BERT-based results, but requires the most GPU memory and time.
LoRA is the most training-efficient adapter method with a reasonable accuracy trade-off.
QLoRA's main advantage (smaller adapter footprint) is more pronounced on smaller GPUs
(<8 GB VRAM) not available in this T4 run; on T4 the memory savings are modest compared
to LoRA. The full-mode run (all 14 k training sentences) is expected to further close the
gap between adapter methods and Full FT.
